# Data Preparation: Clinical Trial Screening Dataset

This notebook prepares the oncology trial screening dataset for model training:
1. Load raw data
2. Filter to most recent note per patient
3. Create STATUS labels based on priority rules
4. Verify data quality and class distribution
5. Create stratified train/validation/test split (70/15/15)
6. Save processed dataset

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Imports successful!")

## Step 1: Load Raw Data

In [ ]:
# For local execution
data_path = '../data/oncology_trial_screening_v02.csv'

# For Google Colab (uncomment and use if needed)
# from google.colab import drive
# drive.mount('/content/drive')
# data_path = '/content/drive/My Drive/path/to/oncology_trial_screening_v02.csv'

df_raw = pd.read_csv(data_path)

print(f"Dataset shape: {df_raw.shape}")
print(f"\nColumn names:\n{df_raw.columns.tolist()}")
print(f"\nFirst few rows:")
df_raw.head()

## Step 2: Data Quality Check

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df_raw.isnull().sum())

print(f"\nData types:")
print(df_raw.dtypes)

# Check date format
print(f"\nDate range: {df_raw['note_date'].min()} to {df_raw['note_date'].max()}")

# Unique patients
print(f"\nTotal unique patients: {df_raw['patient_id'].nunique()}")
print(f"Total notes: {len(df_raw)}")
print(f"Average notes per patient: {len(df_raw) / df_raw['patient_id'].nunique():.2f}")

## Step 3: Filter to Most Recent Note per Patient

In [ ]:
# Convert note_date to datetime if not already
df_raw['note_date'] = pd.to_datetime(df_raw['note_date'])

# Sort by patient_id and note_date to ensure proper ordering
df_raw = df_raw.sort_values(['patient_id', 'note_date'])

# Get most recent note per patient
df_latest = df_raw.loc[df_raw.groupby('patient_id')['note_date'].idxmax()].reset_index(drop=True)

print(f"Dataset after filtering to latest note per patient:")
print(f"Shape: {df_latest.shape}")
print(f"\nVerification - unique patients: {df_latest['patient_id'].nunique()}")
print(f"\nFirst few rows:")
df_latest.head()

## Step 4: Create STATUS Label

Priority rules:
1. If `deceased = 1` → 'DECEASED'
2. Elif `lost_to_follow_up = 1` → 'CENSORED'
3. Elif `available = 1 and eligible = 1` → 'AVAILABLE'
4. Elif `available = 0 and eligible = 1` → 'ELIGIBLE'
5. Else (`eligible = 0`) → 'INELIGIBLE'

In [ ]:
def create_status_label(row):
    """
    Create STATUS label based on priority rules.
    
    Args:
        row: A row from the dataframe with columns: deceased, lost_to_follow_up, available, eligible
    
    Returns:
        Status label as string
    """
    if row['deceased'] == 1:
        return 'DECEASED'
    elif row['lost_to_follow_up'] == 1:
        return 'CENSORED'
    elif row['available'] == 1 and row['eligible'] == 1:
        return 'AVAILABLE'
    elif row['available'] == 0 and row['eligible'] == 1:
        return 'ELIGIBLE'
    else:  # eligible == 0
        return 'INELIGIBLE'

# Apply the function to create STATUS column
df_latest['STATUS'] = df_latest.apply(create_status_label, axis=1)

print("STATUS label created successfully!")
print(f"\nSample rows with STATUS:")
print(df_latest[['patient_id', 'deceased', 'lost_to_follow_up', 'available', 'eligible', 'STATUS']].head(10))

## Step 5: Verify Class Distribution

In [ ]:
# Class distribution
status_counts = df_latest['STATUS'].value_counts()
status_percentages = df_latest['STATUS'].value_counts(normalize=True) * 100

print("Class Distribution:")
print("="*50)
for status in status_counts.index:
    count = status_counts[status]
    percentage = status_percentages[status]
    print(f"{status:15} : {count:4} ({percentage:5.1f}%)")
print("="*50)
print(f"{'TOTAL':15} : {len(df_latest):4}")

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
status_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Class Distribution (Count)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(status_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
colors = sns.color_palette('husl', len(status_counts))
axes[1].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%', 
            colors=colors, startangle=90)
axes[1].set_title('Class Distribution (Percentage)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'class_distribution.png'")

## Step 6: Create Stratified Train/Validation/Test Split

Split ratio: 70% train, 15% validation, 15% test
Stratification: By STATUS class to maintain distribution

In [ ]:
# Create stratified split: first split into train (70%) and temp (30%)
df_train, df_temp = train_test_split(
    df_latest,
    test_size=0.30,
    random_state=42,
    stratify=df_latest['STATUS']
)

# Then split temp into validation (50% of 30% = 15%) and test (50% of 30% = 15%)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp['STATUS']
)

print(f"Train set size: {len(df_train)} ({len(df_train)/len(df_latest)*100:.1f}%)")
print(f"Validation set size: {len(df_val)} ({len(df_val)/len(df_latest)*100:.1f}%)")
print(f"Test set size: {len(df_test)} ({len(df_test)/len(df_latest)*100:.1f}%)")
print(f"Total: {len(df_train) + len(df_val) + len(df_test)}")

# Verify stratification
print("\n" + "="*70)
print("Class distribution across splits:")
print("="*70)

splits_data = {
    'Train': df_train['STATUS'].value_counts(normalize=True) * 100,
    'Validation': df_val['STATUS'].value_counts(normalize=True) * 100,
    'Test': df_test['STATUS'].value_counts(normalize=True) * 100,
    'Overall': df_latest['STATUS'].value_counts(normalize=True) * 100
}

splits_df = pd.DataFrame(splits_data).round(1)
print(splits_df)

# Visualize split distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Class Distribution Across Train/Val/Test Splits', fontsize=14, fontweight='bold', y=1.00)

split_names = ['Train', 'Validation', 'Test', 'Overall']
datasets = [df_train, df_val, df_test, df_latest]

for idx, (ax, split_name, dataset) in enumerate(zip(axes.flatten(), split_names, datasets)):
    status_dist = dataset['STATUS'].value_counts()
    colors = sns.color_palette('husl', len(status_dist))
    ax.bar(status_dist.index, status_dist.values, color=colors)
    ax.set_title(f'{split_name} ({len(dataset)} samples)', fontweight='bold')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    
    # Add count labels on bars
    for i, v in enumerate(status_dist.values):
        ax.text(i, v + 2, str(v), ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('split_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'split_distribution.png'")

## Step 7: Prepare Final Dataset

In [ ]:
# Add split column to original dataframe for reference
df_latest['split'] = 'unknown'
df_latest.loc[df_train.index, 'split'] = 'train'
df_latest.loc[df_val.index, 'split'] = 'validation'
df_latest.loc[df_test.index, 'split'] = 'test'

# Select relevant columns for final dataset
final_columns = ['patient_id', 'note_date', 'cancer_indication', 'free_text_note', 
                  'eligible', 'available', 'deceased', 'lost_to_follow_up', 'STATUS', 'split']

df_processed = df_latest[final_columns].copy()

print(f"Final processed dataset shape: {df_processed.shape}")
print(f"\nColumns: {df_processed.columns.tolist()}")
print(f"\nFirst few rows:")
df_processed.head()

## Step 8: Save Processed Dataset

In [ ]:
# Save the full processed dataset
output_path = 'processed_data.csv'
df_processed.to_csv(output_path, index=False)
print(f"✓ Full processed dataset saved to: {output_path}")

# Also save splits separately for convenience
df_train.to_csv('train_data.csv', index=False)
df_val.to_csv('validation_data.csv', index=False)
df_test.to_csv('test_data.csv', index=False)
print(f"✓ Train set saved to: train_data.csv")
print(f"✓ Validation set saved to: validation_data.csv")
print(f"✓ Test set saved to: test_data.csv")

# Save class mapping for reference
class_mapping = {status: idx for idx, status in enumerate(sorted(df_latest['STATUS'].unique()))}
print(f"\nClass mapping:")
for status, idx in sorted(class_mapping.items()):
    print(f"  {status:15} → {idx}")

# Save mapping to file
import json
with open('class_mapping.json', 'w') as f:
    json.dump(class_mapping, f, indent=2)
print(f"\n✓ Class mapping saved to: class_mapping.json")

## Summary

In [ ]:
print("\n" + "="*70)
print("DATA PREPARATION SUMMARY")
print("="*70)
print(f"\n✓ Loaded raw dataset: {df_raw.shape[0]} rows")
print(f"✓ Filtered to latest note per patient: {df_latest.shape[0]} rows")
print(f"✓ Created STATUS labels using priority rules")
print(f"\nClass Distribution:")
for status in sorted(df_latest['STATUS'].unique()):
    count = (df_latest['STATUS'] == status).sum()
    percentage = count / len(df_latest) * 100
    print(f"  {status:15} : {count:4} ({percentage:5.1f}%)")

print(f"\n✓ Created stratified train/val/test split:")
print(f"  Train   : {len(df_train):4} samples (70%)")
print(f"  Validation: {len(df_val):4} samples (15%)")
print(f"  Test    : {len(df_test):4} samples (15%)")

print(f"\n✓ Saved processed datasets:")
print(f"  - processed_data.csv (full dataset with split labels)")
print(f"  - train_data.csv")
print(f"  - validation_data.csv")
print(f"  - test_data.csv")
print(f"  - class_mapping.json")

print(f"\n✓ Generated visualizations:")
print(f"  - class_distribution.png")
print(f"  - split_distribution.png")

print("\n" + "="*70)
print("Ready for model training! Proceed to notebook 02 or 03.")
print("="*70)